In [ ]:
import torch
from occhio.autoencoder import TiedLinearRelu
from occhio.distributions.hierarchical import HierarchicalSparse
from occhio.model_grid import ModelGrid, Axis
from occhio.toy_model import ToyModel
from occhio.visualization import *

device = "mps"

In [ ]:
N_FEATURES, N_HIDDEN = 5, 2

axis_embed = Axis(label="p_base", values=torch.logspace(-1, 0, 6))


def create_embedding_model(params):
    return ToyModel(
        distribution=HierarchicalSparse(
            N_FEATURES,
            p_base=float(params["p_base"]),
            depth_decay=0.85,
            max_children=3,
            device=device,
            generator=torch.Generator(device=device).manual_seed(42),
        ),
        importances=0.9 ** torch.arange(N_FEATURES),
        ae=TiedLinearRelu(
            N_FEATURES,
            N_HIDDEN,
            device=device,
            generator=torch.Generator(device=device).manual_seed(7),
        ),
    )


grid_embed = ModelGrid(create_embedding_model, axes=[axis_embed])

In [ ]:
grid_embed.fit(batch_size=2048, n_epochs=20_000)

In [ ]:
print(axis_embed.values)
plot_embedding(grid_embed)

In [ ]:
N_FEATURES, N_HIDDEN = 5, 2

axis_importance = Axis(label="Importance", values=torch.logspace(-1, 1, 70))
axis_density = Axis(label="p_base", values=torch.logspace(-1, 0, 70))


def create_phase_model(params):
    return ToyModel(
        distribution=HierarchicalSparse(
            N_FEATURES,
            p_base=float(params["p_base"]),
            depth_decay=0.85,
            max_children=3,
            device=device,
            generator=torch.Generator(device=device).manual_seed(42),
        ),
        importances=float(params["Importance"]) ** torch.arange(N_FEATURES),
        ae=TiedLinearRelu(
            N_FEATURES,
            N_HIDDEN,
            device=device,
            generator=torch.Generator(device=device).manual_seed(7),
        ),
    )


grid_phase = ModelGrid(
    create_phase_model, axes=[axis_importance, axis_density], cache_samples=True
)

In [ ]:
grid_phase.fit(batch_size=216, n_epochs=10_000)

In [ ]:
import pickle

# Save the fitted grid_phase to disk for reuse later
with open("grid_phase.pkl", "wb") as f:
    pickle.dump(grid_phase, f)

# You can load and reconstruct the object like this:
with open("grid_phase.pkl", "rb") as f:
    grid_phase_loaded = pickle.load(f)
    # Now grid_phase_loaded is a reconstructed ModelGrid object

In [ ]:
# This feature is always active
plot_phase_change(grid_phase, tracked_feature=0)

In [ ]:
plot_phase_change(grid_phase, tracked_feature=1)  # depth-1 child

In [ ]:
plot_phase_change(grid_phase, tracked_feature=2)

In [ ]:
plot_phase_change(grid_phase, tracked_feature=3)

In [ ]:
plot_phase_change(grid_phase, tracked_feature=4)  # depth-2 leaf — sparsest

In [ ]:
# Exp 3 — Geometry: 100 features, 20 hidden (5:1), sweep p_base
# Flat importance so geometry is driven by density alone
N_FEATURES, N_HIDDEN = 100, 20

axis_geo = Axis(label="p_base", values=torch.logspace(-2, 0, 16))


def create_geometry_model(params):
    return ToyModel(
        distribution=HierarchicalSparse(
            N_FEATURES,
            p_base=float(params["p_base"]),
            depth_decay=0.85,
            max_children=4,
            device=device,
            generator=torch.Generator(device=device).manual_seed(42),
        ),
        importances=0.999 ** torch.arange(N_FEATURES),
        ae=TiedLinearRelu(
            N_FEATURES,
            N_HIDDEN,
            device=device,
            generator=torch.Generator(device=device).manual_seed(7),
        ),
    )


grid_geo = ModelGrid(create_geometry_model, axes=[axis_geo])

In [ ]:
grid_geo.fit(batch_size=2048, n_epochs=10_000)

In [ ]:
plot_geometry(grid_phase)

In [ ]:
plot_geometry(grid_geo)

In [ ]:
m